# Perhitungan Sentralitas

## Import Library

In [3]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [4]:
# Pengaturan tampilan
pd.set_option('display.max_colwidth', None)

# Float 8 desimal
pd.options.display.float_format = '{:.8f}'.format

# Membaca CSV
df = pd.read_csv('../data/processed/edges.csv')

# Konversi numerik
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])

# Pembulatan
df['Inverse_Weight'] = df['Inverse_Weight'].round(8)

# Urutkan berdasarkan Weight terbesar
df = df.sort_values(
    by='Weight',
    ascending=False
).reset_index(drop=True)

# Output
display(df)

,Guru,Murid,Weight,Inverse_Weight
0,ابن عمر,نافع,11346,0.00008814
1,معمر,عبد الرزاق,6402,0.00015620
2,ابو هشام بن عروه,هشام بن عروه,6390,0.00015649
3,ابن عباس,عكرمه,5456,0.00018328
4,ابن عباس,سعيد بن جبير,4663,0.00021445
...,...,...,...,...
776793,كيسان,محمد بن ربيعه,1,1.00000000
776794,عليا,صفوان بن عيسي الزهري,1,1.00000000
776795,ناسا من الصحابه,جابر بن زيد,1,1.00000000
776796,الزهري,عبد الرزاق بن همام,1,1.00000000


## Load Data

In [5]:
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


In [6]:
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


### Bentuk graph

In [7]:
G = nx.from_pandas_edgelist(
    df,
    source='Murid',
    target='Guru',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

In [8]:
print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 177047
Jumlah edge : 776798


In [9]:
display(df)

,Guru,Murid,Weight,Inverse_Weight
0,ابن عمر,نافع,11346,0.00008814
1,معمر,عبد الرزاق,6402,0.00015620
2,ابو هشام بن عروه,هشام بن عروه,6390,0.00015649
3,ابن عباس,عكرمه,5456,0.00018328
4,ابن عباس,سعيد بن جبير,4663,0.00021445
...,...,...,...,...
776793,كيسان,محمد بن ربيعه,1,1.00000000
776794,عليا,صفوان بن عيسي الزهري,1,1.00000000
776795,ناسا من الصحابه,جابر بن زيد,1,1.00000000
776796,الزهري,عبد الرزاق بن همام,1,1.00000000


In [10]:
# Hapus self-loop
G.remove_edges_from(nx.selfloop_edges(G))

In [11]:
print("Jumlah node setelah hapus self-loop:", G.number_of_nodes())
print("Jumlah edge setelah hapus self-loop:", G.number_of_edges())

Jumlah node setelah hapus self-loop: 177047
Jumlah edge setelah hapus self-loop: 776798


In [12]:
# Degree
degree_dict = dict(G.degree(weight='Weight'))
print("✅ Degree selesai")

# Degree Centrality
deg = nx.degree_centrality(G)
print("✅ Degree Centrality selesai.")

# In-Degree Centrality
if G.is_directed():
    in_deg = nx.in_degree_centrality(G)
    print("✅ In-Degree Centrality selesai.")
else:
    in_deg = {}

# Out-Degree Centrality
if G.is_directed():
    out_deg = nx.out_degree_centrality(G)
    print("✅ Out-Degree Centrality selesai.")
else:
    out_deg = {}

✅ Degree selesai
✅ Degree Centrality selesai.
✅ In-Degree Centrality selesai.
✅ Out-Degree Centrality selesai.


In [13]:
# === Eigenvector Centrality (Power Iteration)
try:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=5000, tol=1e-05, weight='Weight')
    print("✅ Eigenvector Centrality selesai.")
except nx.PowerIterationFailedConvergence:
    eigenvector_centrality = {}

# # === Eigenvector Centrality (NumPy version)
# try:
#     eigenvector_numpy_centrality = nx.eigenvector_centrality_numpy(G, weight='Weight')
#     print("✅ Eigenvector Numpy Centrality selesai.")
# except Exception as e:
#     eigenvector_numpy_centrality = {}
#     print(f"Proses gagal: {e}")

✅ Eigenvector Centrality selesai.


In [16]:
print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 12169
Jumlah edge : 40000


In [17]:
from tqdm import tqdm

# fungsi hitung closeness per node
def closeness_for_node(node):
    return node, nx.closeness_centrality(G, u=node, distance="Inverse_Weight")

# daftar semua node
nodes = list(G.nodes())

closeness = {}
for node in tqdm(nodes, desc="Closeness Centrality"):
    closeness[node] = nx.closeness_centrality(G, u=node, distance="Inverse_Weight")


print("✅ Closeness Centrality selesai.")

Closeness Centrality: 100%|██████████| 12169/12169 [1:02:32<00:00,  3.24it/s]

✅ Closeness Centrality selesai.


In [18]:
import networkx as nx
from tqdm import tqdm

def betweenness_centrality_with_progress(G, normalized=True, weight=None):
    nodes = list(G.nodes())
    bet = dict.fromkeys(nodes, 0.0)

    for s in tqdm(nodes, desc="Betweenness Centrality (Nodes)"):
        # hitung betweenness subset: dari source s ke semua node
        contrib = nx.betweenness_centrality_subset(
            G, sources=[s], targets=nodes,
            normalized=normalized, weight=weight
        )
        # gabungkan kontribusi
        for n, v in contrib.items():
            bet[n] += v

    return bet


# =====================
# Pemakaian
# =====================
bet_node = betweenness_centrality_with_progress(G, normalized=True, weight="Inverse_weight")

print("✅ Betweenness selesai.")

Betweenness Centrality (Nodes): 100%|██████████| 12169/12169 [18:27<00:00, 10.99it/s] 

✅ Betweenness selesai.


In [19]:
# ==============================================
# Tampilkan TOP 20 Perawi untuk setiap centrality
# ==============================================
import pandas as pd

centrality_results = pd.DataFrame({
    'Perawi': list(G.nodes()),
    'Degree': [degree_dict.get(n, 0) for n in G.nodes()],
    'Degree_Centrality': [deg.get(n, 0) for n in G.nodes()],
    'In_Degree_Centrality': [in_deg.get(n, 0) for n in G.nodes()],
    'Out_Degree_Centrality': [out_deg.get(n, 0) for n in G.nodes()],
    'Eigenvector_Centrality': [globals().get('eigenvector_centrality', {}).get(n, 0) for n in G.nodes()],
    'Closeness_Centrality': [globals().get('closeness', {}).get(n, 0) for n in G.nodes()],
    'Betweenness_Centrality': [globals().get('bet_node', {}).get(n, 0) for n in G.nodes()],
})

# Bulatkan untuk tampilan
centrality_results = centrality_results.round(6)

print('\n' + '='*80)
print('TOP 20 — Degree Centrality')
print('='*80)
display(centrality_results.nlargest(20, 'Degree_Centrality')[['Perawi', 'Degree', 'Degree_Centrality']])

print('\n' + '='*80)
print('TOP 20 — In-Degree Centrality (Peran sebagai Guru)')
print('='*80)
display(centrality_results.nlargest(20, 'In_Degree_Centrality')[['Perawi', 'In_Degree_Centrality']])

print('\n' + '='*80)
print('TOP 20 — Out-Degree Centrality (Peran sebagai Murid)')
print('='*80)
display(centrality_results.nlargest(20, 'Out_Degree_Centrality')[['Perawi', 'Out_Degree_Centrality']])

print('\n' + '='*80)
print('TOP 20 — Eigenvector Centrality')
print('='*80)
display(centrality_results.nlargest(20, 'Eigenvector_Centrality')[['Perawi', 'Eigenvector_Centrality']])

print('\n' + '='*80)
print('TOP 20 — Closeness Centrality')
print('='*80)
display(centrality_results.nlargest(20, 'Closeness_Centrality')[['Perawi', 'Closeness_Centrality']])

print('\n' + '='*80)
print('TOP 20 — Betweenness Centrality')
print('='*80)
display(centrality_results.nlargest(20, 'Betweenness_Centrality')[['Perawi', 'Betweenness_Centrality']])

# Simpan jika ingin
# centrality_results.to_csv('../data/processed/centrality_top20_notebook.csv', index=False, encoding='utf-8-sig')


TOP 20 — Degree Centrality


,Perawi,Degree,Degree_Centrality
10,ابو هريره,52846,0.02722500
33,سفيان,48556,0.02645100
21,شعبه,50261,0.02440000
7,ابن عباس,39321,0.02039000
14,عائشه,32693,0.01731200
24,الاعمش,34730,0.01724400
1,ابن عمر,27935,0.01718800
25,ابو عبد الله الحافظ,15860,0.01712500
11,الزهري,42075,0.01706900
75,عبد الله,17829,0.01610300



TOP 20 — In-Degree Centrality (Peran sebagai Guru)


,Perawi,In_Degree_Centrality
10,ابو هريره,0.01992700
7,ابن عباس,0.01504100
14,عائشه,0.01317200
33,سفيان,0.01295100
44,انس بن مالك,0.01282200
1,ابن عمر,0.01269200
21,شعبه,0.01198600
24,الاعمش,0.00986700
200,علي,0.00956200
656,رجل,0.00894100



TOP 20 — Out-Degree Centrality (Peran sebagai Murid)


...

,Perawi,Eigenvector_Centrality
10,ابو هريره,0.51953900
14,عائشه,0.48360300
1,ابن عمر,0.38208000
7,ابن عباس,0.26587700
18,انس,0.17227900
179,عمر,0.15470300
44,انس بن مالك,0.12854700
32,سعيد بن المسيب,0.12310100
11,الزهري,0.11976400
13,عروه,0.11489300



TOP 20 — Closeness Centrality


,Perawi,Closeness_Centrality
10,ابو هريره,7.42770200
14,عائشه,7.39894000
7,ابن عباس,7.30808000
179,عمر,7.29878600
439,عمر بن الخطاب,7.27630300
1,ابن عمر,7.23744200
44,انس بن مالك,7.22137600
18,انس,7.19483200
32,سعيد بن المسيب,7.15289900
17,قتاده,7.10028600



TOP 20 — Betweenness Centrality


,Perawi,Betweenness_Centrality
10,ابو هريره,0.04871200
21,شعبه,0.04602100
33,سفيان,0.04362400
25,ابو عبد الله الحافظ,0.03204900
14,عائشه,0.02840400
192,ابو بكر,0.02289300
54,ابو داود,0.02204700
7,ابن عباس,0.02145200
11,الزهري,0.02133900
75,عبد الله,0.02091100


### Menghitung Degree, Degree Centrality, In Degree, Out Degree, Eigenvector, Closeness, dan Betweenness

In [28]:
# ============================================
# Tambahkan centrality metrics sebagai node attributes
# CATATAN: Ambil dari dictionary asli (177k nodes) bukan dari G (12k nodes)
# ============================================
for node in G.nodes():
    # Ambil dari dictionary asli, jika ada di graph subset
    G.nodes[node]['degree'] = degree_dict.get(node, 0)
    G.nodes[node]['degree_centrality'] = deg.get(node, 0)
    G.nodes[node]['in_degree_centrality'] = in_deg.get(node, 0)
    G.nodes[node]['out_degree_centrality'] = out_deg.get(node, 0)
    G.nodes[node]['eigenvector_centrality'] = eigenvector_centrality.get(node, 0)
    # Closeness dan Betweenness hanya untuk subset 40k
    G.nodes[node]['closeness_centrality'] = closeness.get(node, 0)
    G.nodes[node]['betweenness_centrality'] = bet_node.get(node, 0)

print("✅ Node attributes ditambahkan")

# ============================================
# Ekspor ke GraphML
# ============================================
output_path = '../data/processed/Perawi.graphml'
nx.write_graphml(G, output_path)
print(f"✅ Graph berhasil diekspor ke: {output_path}")
print(f"   Nodes: {G.number_of_nodes()}")
print(f"   Edges: {G.number_of_edges()}")
print(f"\n📊 Node Attributes:")
print(f"   - degree, degree_centrality, eigenvector = dari SEMUA data (177k)")
print(f"   - closeness, betweenness = dari 40k data saja")

✅ Node attributes ditambahkan
✅ Graph berhasil diekspor ke: ../data/processed/Perawi.graphml
   Nodes: 12169
   Edges: 40000

📊 Node Attributes:
   - degree, degree_centrality, eigenvector = dari SEMUA data (177k)
   - closeness, betweenness = dari 40k data saja


In [29]:
# ============================================
# VERIFIKASI: Cek berapa banyak node di setiap centrality metric
# ============================================
print("="*70)
print("VERIFIKASI JUMLAH NODE UNTUK SETIAP CENTRALITY METRIC")
print("="*70)
print(f"\nGraph saat ini:")
print(f"  Total Nodes: {G.number_of_nodes()}")
print(f"  Total Edges: {G.number_of_edges()}")

print(f"\nCentrality Metrics yang tersedia:")
print(f"  • Degree: {len(degree_dict)} node")
print(f"  • Degree Centrality: {len(deg)} node")
print(f"  • In-Degree Centrality: {len(in_deg)} node")
print(f"  • Out-Degree Centrality: {len(out_deg)} node")
print(f"  • Eigenvector Centrality: {len(eigenvector_centrality)} node")
print(f"  • Closeness Centrality: {len(closeness)} node ⚠️  (40k data saja)")
print(f"  • Betweenness Centrality: {len(bet_node)} node ⚠️  (40k data saja)")

print(f"\n⚠️  PENTING:")
print(f"  - Degree, Eigenvector dihitung dari SEMUA DATA (besar)")
print(f"  - Closeness, Betweenness hanya dari 40,000 edges pertama (lebih cepat)")
print(f"  - Di GraphML: node tanpa closeness/betweenness akan bernilai 0")

VERIFIKASI JUMLAH NODE UNTUK SETIAP CENTRALITY METRIC

Graph saat ini:
  Total Nodes: 12169
  Total Edges: 40000

Centrality Metrics yang tersedia:
  • Degree: 177047 node
  • Degree Centrality: 177047 node
  • In-Degree Centrality: 177047 node
  • Out-Degree Centrality: 177047 node
  • Eigenvector Centrality: 177047 node
  • Closeness Centrality: 12169 node ⚠️  (40k data saja)
  • Betweenness Centrality: 12169 node ⚠️  (40k data saja)

⚠️  PENTING:
  - Degree, Eigenvector dihitung dari SEMUA DATA (besar)
  - Closeness, Betweenness hanya dari 40,000 edges pertama (lebih cepat)
  - Di GraphML: node tanpa closeness/betweenness akan bernilai 0


In [ ]:
# ============================================
# EXPORT DENGAN SEMUA DATA (bukan 40k saja)
# ============================================
print("="*70)
print("REBUILD GRAPH DARI SEMUA DATA")
print("="*70)

# Buat graph baru dari df (semua data)
G_all = nx.from_pandas_edgelist(
    df,
    source='Murid',
    target='Guru',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

# Hapus self-loop
G_all.remove_edges_from(nx.selfloop_edges(G_all))

print(f"Graph dari SEMUA data:")
print(f"  Nodes: {G_all.number_of_nodes()}")
print(f"  Edges: {G_all.number_of_edges()}")

# ============================================
# Tambahkan centrality metrics sebagai node attributes
# ============================================
for node in G_all.nodes():
    G_all.nodes[node]['degree'] = degree_dict.get(node, 0)
    G_all.nodes[node]['degree_centrality'] = deg.get(node, 0)
    G_all.nodes[node]['in_degree_centrality'] = in_deg.get(node, 0)
    G_all.nodes[node]['out_degree_centrality'] = out_deg.get(node, 0)
    G_all.nodes[node]['eigenvector_centrality'] = eigenvector_centrality.get(node, 0)
    # Closeness dan Betweenness hanya untuk 40k subset (jika ada)
    G_all.nodes[node]['closeness_centrality'] = closeness.get(node, 0)
    G_all.nodes[node]['betweenness_centrality'] = bet_node.get(node, 0)

print("✅ Node attributes ditambahkan")

# ============================================
# Ekspor ke GraphML (SEMUA DATA)
# ============================================
output_path_all = '../data/processed/CentralityPerawi_All.graphml'
nx.write_graphml(G_all, output_path_all)
print(f"✅ Graph SEMUA DATA berhasil diekspor ke: {output_path_all}")
print(f"   Nodes: {G_all.number_of_nodes()}")
print(f"   Edges: {G_all.number_of_edges()}")
print(f"\n📊 Node Attributes:")
print(f"   - degree, degree_centrality, eigenvector = dari SEMUA data ✅")
print(f"   - closeness, betweenness = dari 40k subset (ada/tidak ada tergantung node)")
print(f"\nFile untuk Gephi: CentralityPerawi_All.graphml")